# Base

In [1]:
# !pip install ipywidgets --upgrade
# !pip install notebook --upgrade
# !jupyter nbextension enable --py widgetsnbextension

In [4]:
# !pip install langchain_community

In [1]:
from ragatouille import RAGPretrainedModel

RAG = RAGPretrainedModel.from_pretrained("colbert-ir/colbertv2.0")

/Users/Sankalp.C/anaconda3/envs/clincodex-1/lib/python3.9/site-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


[Dec 28, 11:29:34] Loading segmented_maxsim_cpp extension (set COLBERT_LOAD_TORCH_EXTENSION_VERBOSE=True for more info)...


/Users/Sankalp.C/anaconda3/envs/clincodex-1/lib/python3.9/site-packages/torch/cuda/amp/grad_scaler.py:126: UserWarning: torch.cuda.amp.GradScaler is enabled, but CUDA is not available.  Disabling.
  warnings.warn(


# Rag Pipe

In [17]:
import pickle
import os

from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("/Users/Sankalp.C/Desktop/Neo/clincodex/data/ICD10CM_Coding_Guide.pdf")
pages = loader.load_and_split()

len(pages)

125

In [18]:
code_doc = ""

for page in pages:
  code_doc += page.page_content

print(code_doc)

ICD-10-CM Official Guidelines for Coding and Reporting 
FY 2024 -- UPDATED April 1, 2024 
(April 1, 2024 - September 30, 2024) 
 
Narrative changes appear in bold text 
Items underlined have been moved within the guidelines since the October 2024, FY 2024 version 
Italics are used to indicate revisions to heading changes 
 
The Centers for Medicare and Medicaid Services (CMS) and the National Center for Health 
Statistics (NCHS), two departments within the U.S. Federal Government’s Department of Health 
and Human Services (DHHS) provide the following guidelines for coding and reporting using the 
International Classification of Diseases, 10th Revision, Clinical Modification (ICD-10-CM). These 
guidelines should be used as a companion document to the official version of the ICD -10-CM as 
published on the NCHS website. The ICD-10-CM is a morbidity classification published by the 
United States for classifying diagnoses and reason for visits in all he alth care settings. The ICD-
10-CM i

In [19]:
len(code_doc)

314288

In [20]:
import re
def thorough_clean_text(page_text: str) -> str:
        # 1) Remove repeated noise (headers, footers, disclaimers).
        patterns_to_remove = [
            r"(?i)FY\s*2024\s*ICD-10-CM\s*Official\s*Guidelines.*",  # Example header
            r"(?i)Centers\s*for\s*Medicare\s*&\s*Medicaid\s*Services.*",
            r"Page\s*\d+\s+of\s+\d+",                               # Typical footer
            r"(?i)Updated\s*\d{2}/\d{2}/\d{4}",                     # e.g., "Updated 02/01/2024"
        ]
        cleaned_text = page_text
        for pattern in patterns_to_remove:
            cleaned_text = re.sub(pattern, "", cleaned_text, flags=re.MULTILINE)

        # 2) Fix hyphenated words broken across lines (e.g., "classi-\nfication" => "classification").
        cleaned_text = re.sub(r"(\w)-\n(\w)", r"\1\2", cleaned_text)

        # 3) Convert multiple line breaks into a single line break.
        cleaned_text = re.sub(r"\n+", "\n", cleaned_text)

        # 4) Trim leading/trailing whitespace.
        cleaned_text = cleaned_text.strip()

        # 5) Merge lines into coherent paragraphs based on punctuation & length heuristics.
        lines = cleaned_text.split("\n")
        merged_lines = []
        paragraph_buffer = []

        for line in lines:
            line = line.strip()
            # Blank line => paragraph boundary
            if not line:
                if paragraph_buffer:
                    merged_lines.append(" ".join(paragraph_buffer))
                    paragraph_buffer = []
                continue

            # If line doesn't end with common sentence punctuation or is very short,
            # we assume it's a continuation of the same paragraph.
            if (len(line) < 80) or (not re.search(r"[.?!;:,]\s*$", line)):
                paragraph_buffer.append(line)
            else:
                paragraph_buffer.append(line)
                merged_lines.append(" ".join(paragraph_buffer))
                paragraph_buffer = []

        # Append any leftover text in the paragraph buffer
        if paragraph_buffer:
            merged_lines.append(" ".join(paragraph_buffer))

        # Reassemble paragraphs with double newlines
        final_cleaned = "\n\n".join(merged_lines).strip()
        return final_cleaned

code_doc = thorough_clean_text(code_doc) 

In [21]:
len(code_doc)

307038

In [24]:
print(code_doc)

ICD-10-CM Official Guidelines for Coding and Reporting FY 2024 -- UPDATED April 1, 2024 (April 1, 2024 - September 30, 2024)

Narrative changes appear in bold text Items underlined have been moved within the guidelines since the October 2024, FY 2024 version Italics are used to indicate revisions to heading changes

The Centers for Medicare and Medicaid Services (CMS) and the National Center for Health Statistics (NCHS), two departments within the U.S. Federal Government’s Department of Health and Human Services (DHHS) provide the following guidelines for coding and reporting using the International Classification of Diseases, 10th Revision, Clinical Modification (ICD-10-CM). These guidelines should be used as a companion document to the official version of the ICD -10-CM as published on the NCHS website. The ICD-10-CM is a morbidity classification published by the United States for classifying diagnoses and reason for visits in all he alth care settings. The ICD10-CM is based on the I

In [ ]:
# Regex pattern
pattern = r"(?<=1\. Chapter 1: Certain Infectious and Parasitic Diseases).*?(?=Section II\. Selection of Principal Diagnosis)"

matches = list(re.finditer(pattern, code_doc, re.DOTALL))

# Get the second match if it exists
if len(matches) >= 2:
    second_match = matches[1].group().strip()  # Use .strip() to remove leading/trailing whitespace
    print("Second Match:")
    print("Chapter 1: Certain Infectious and Parasitic Diseases " + second_match)
    code_guide = "1. Chapter 1: Certain Infectious and Parasitic Diseases " + second_match
else:
    print("Less than two matches found.")

Second Match:
Chapter 1: Certain Infectious and Parasitic Diseases (A00-B99), U07.1, U09.9 a. Human Immunodeficiency Virus (HIV) Infections 1) Code only confirmed cases Code only confirmed cases of HIV infection/illness. This is an exception to the hospital inpatient guideline Section II, H.

In this context, “confirmation” does not require documentation of positive serology or culture for HIV; the provider’s diagnostic statement that the patient is HIV positive or has an HIV-related illness is sufficient. 2) Selection and sequencing of HIV codes (a) Patient admitted for HIV-related condition If a patient is admitted for an HIV-related condition, the principal diagnosis should be B20, Human immunodeficiency virus [HIV] disease followed by additional diagnosis codes for all reported HIV-related conditions.ICD-10-CM Official Guidelines for Coding and Reporting FY 2024

An exception to this guideline is if the reason for admission is hemolytic-uremic syndrome associated with HIV disease. 

In [34]:
# Regular expression pattern
pattern = r"\d+\.\sChapter\s\d+:"

# Split the text based on the pattern
segments = re.split(f"({pattern})", code_guide)

# Combine the matched patterns with their corresponding text
code_guide_chapters = ["".join(x) for x in zip(segments[1::2], segments[2::2])]

# # Print the chapters
# for i, chapter in enumerate(chapters, 1):
#     print(f"Chapter {i}:\n{chapter}\n")

code_guide_chapters

['1. Chapter 1: Certain Infectious and Parasitic Diseases (A00-B99), U07.1, U09.9 a. Human Immunodeficiency Virus (HIV) Infections 1) Code only confirmed cases Code only confirmed cases of HIV infection/illness. This is an exception to the hospital inpatient guideline Section II, H.\n\nIn this context, “confirmation” does not require documentation of positive serology or culture for HIV; the provider’s diagnostic statement that the patient is HIV positive or has an HIV-related illness is sufficient. 2) Selection and sequencing of HIV codes (a) Patient admitted for HIV-related condition If a patient is admitted for an HIV-related condition, the principal diagnosis should be B20, Human immunodeficiency virus [HIV] disease followed by additional diagnosis codes for all reported HIV-related conditions.ICD-10-CM Official Guidelines for Coding and Reporting FY 2024\n\nAn exception to this guideline is if the reason for admission is hemolytic-uremic syndrome associated with HIV disease. Assig

In [48]:
code_guide_ids= [
    "A00-B99, U07.1, U09.9",
    "C00-D49",
    "D50-D89",
    "E00-E89",
    "F01-F99",
    "G00-G99",
    "H00-H59",
    "H60-H95",
    "I00-I99",
    "J00-J99, U07.0",
    "K00-K95",
    "L00-L99",
    "M00-M99",
    "N00-N99",
    "O00-O9A",
    "P00-P96",
    "Q00-Q99",
    "R00-R99",
    "S00-T88",
    "V00-Y99",
    "Z00-Z99",
    "U00-U85",
]

len(code_guide_ids)

22

In [47]:
code_guide_meta = chapters = [
    {
        "entity": "chapter",
        "index": 1,
        "id": "A00-B99, U07.1, U09.9",
        "description": "Certain Infectious and Parasitic Diseases (A00-B99), U07.1, U09.9",
    },
    {
        "entity": "chapter",
        "index": 2,
        "id": "C00-D49",
        "description": "Neoplasms (C00-D49)",
    },
    {
        "entity": "chapter",
        "index": 3,
        "id": "D50-D89",
        "description": "Diseases of the blood and blood-forming organs and certain disorders involving the immune mechanism (D50-D89)",
    },
    {
        "entity": "chapter",
        "index": 4,
        "id": "E00-E89",
        "description": "Endocrine, Nutritional, and Metabolic Diseases (E00-E89)",
    },
    {
        "entity": "chapter",
        "index": 5,
        "id": "F01-F99",
        "description": "Mental, Behavioral and Neurodevelopmental Disorders (F01-F99)",
    },
    {
        "entity": "chapter",
        "index": 6,
        "id": "G00-G99",
        "description": "Diseases of the Nervous System (G00-G99)",
    },
    {
        "entity": "chapter",
        "index": 7,
        "id": "H00-H59",
        "description": "Diseases of the Eye and Adnexa (H00-H59)",
    },
    {
        "entity": "chapter",
        "index": 8,
        "id": "H60-H95",
        "description": "Diseases of the Ear and Mastoid Process (H60-H95)",
    },
    {
        "entity": "chapter",
        "index": 9,
        "id": "I00-I99",
        "description": "Diseases of the Circulatory System (I00-I99)",
    },
    {
        "entity": "chapter",
        "index": 10,
        "id": "J00-J99, U07.0",
        "description": "Diseases of the Respiratory System (J00-J99), U07.0",
    },
    {
        "entity": "chapter",
        "index": 11,
        "id": "K00-K95",
        "description": "Diseases of the Digestive System (K00-K95)",
    },
    {
        "entity": "chapter",
        "index": 12,
        "id": "L00-L99",
        "description": "Diseases of the Skin and Subcutaneous Tissue (L00-L99)",
    },
    {
        "entity": "chapter",
        "index": 13,
        "id": "M00-M99",
        "description": "Diseases of the Musculoskeletal System and Connective Tissue (M00-M99)",
    },
    {
        "entity": "chapter",
        "index": 14,
        "id": "N00-N99",
        "description": "Diseases of the Genitourinary System (N00-N99)",
    },
    {
        "entity": "chapter",
        "index": 15,
        "id": "O00-O9A",
        "description": "Pregnancy, Childbirth, and the Puerperium (O00-O9A)",
    },
    {
        "entity": "chapter",
        "index": 16,
        "id": "P00-P96",
        "description": "Certain Conditions Originating in the Perinatal Period (P00-P96)",
    },
    {
        "entity": "chapter",
        "index": 17,
        "id": "Q00-Q99",
        "description": "Congenital malformations, deformations, and chromosomal abnormalities (Q00-Q99)",
    },
    {
        "entity": "chapter",
        "index": 18,
        "id": "R00-R99",
        "description": "Symptoms, signs, and abnormal clinical and laboratory findings, not elsewhere classified (R00-R99)",
    },
    {
        "entity": "chapter",
        "index": 19,
        "id": "S00-T88",
        "description": "Injury, poisoning, and certain other consequences of external causes (S00-T88)",
    },
    {
        "entity": "chapter",
        "index": 20,
        "id": "V00-Y99",
        "description": "External Causes of Morbidity (V00-Y99)",
    },
    {
        "entity": "chapter",
        "index": 21,
        "id": "Z00-Z99",
        "description": "Factors influencing health status and contact with health services (Z00-Z99)",
    },
    {
        "entity": "chapter",
        "index": 22,
        "id": "U00-U85",
        "description": "Codes for Special Purposes (U00-U85)",
    },
]

len(code_guide_meta)

22

In [50]:
RAG.index(
    collection=code_guide_chapters,
    index_name="icd_guide_24_section_1_chapter_c",
    max_document_length=512,
    split_documents=False,
    document_ids=code_guide_ids,
    document_metadatas=code_guide_meta
)

New index_name received! Updating current index_name (icd_guide_24_section_1_chapter_c) to icd_guide_24_section_1_chapter_c
---- WARNING! You are using PLAID with an experimental replacement for FAISS for greater compatibility ----
This is a behaviour change from RAGatouille 0.8.0 onwards.
This works fine for most users and smallish datasets, but can be considerably slower than FAISS and could cause worse results in some situations.
If you're confident with FAISS working on your machine, pass use_faiss=True to revert to the FAISS-using behaviour.
--------------------


[Dec 28, 12:42:54] #> Note: Output directory .ragatouille/colbert/indexes/icd_guide_24_section_1_chapter_c already exists




/Users/Sankalp.C/anaconda3/envs/clincodex-1/lib/python3.9/site-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


[Dec 28, 12:42:58] [0] 		 #> Encoding 22 passages..


/Users/Sankalp.C/anaconda3/envs/clincodex-1/lib/python3.9/site-packages/torch/cuda/amp/grad_scaler.py:126: UserWarning: torch.cuda.amp.GradScaler is enabled, but CUDA is not available.  Disabling.
  warnings.warn(
/Users/Sankalp.C/anaconda3/envs/clincodex-1/lib/python3.9/site-packages/torch/amp/autocast_mode.py:250: UserWarning: User provided device_type of 'cuda', but CUDA is not available. Disabling
  warnings.warn(
100%|██████████| 1/1 [00:16<00:00, 16.56s/it]

[Dec 28, 12:43:14] [0] 		 avg_doclen_est = 359.68182373046875 	 len(local_sample) = 22
[Dec 28, 12:43:14] [0] 		 Creating 1,024 partitions.
[Dec 28, 12:43:14] [0] 		 *Estimated* 7,913 embeddings.
[Dec 28, 12:43:14] [0] 		 #> Saving the indexing plan to .ragatouille/colbert/indexes/icd_guide_24_section_1_chapter_c/plan.json ..


...

[Dec 28, 12:43:35] #> Optimizing IVF to store map from centroids to list of pids..
[Dec 28, 12:43:35] #> Building the emb2pid mapping..
[Dec 28, 12:43:35] len(emb2pid) = 7913



100%|██████████| 1024/1024 [00:00<00:00, 37941.07it/s]

[Dec 28, 12:43:35] #> Saved optimized IVF to .ragatouille/colbert/indexes/icd_guide_24_section_1_chapter_c/ivf.pid.pt


Done indexing!


'.ragatouille/colbert/indexes/icd_guide_24_section_1_chapter_c'

In [67]:
results = RAG.search(query='''Type 2 Diabetes Mellitus || E11 ''', k=3)

In [68]:
results

[{'content': '4. Chapter 4: Endocrine, Nutritional, and Metabolic Diseases (E00-E89) a. Diabetes mellitus The diabetes mellitus codes are combination codes that include the type of diabetes mellitus, the body system affected, and the complications affecting that body system. As many codes within a particular category as are necessary to describe all of the complications of the disease may be used. They should be sequenced based on the reason for a particular encounter. Assign as many codesICD-10-CM Official Guidelines for Coding and Reporting FY 2024\n\nfrom categories E08 – E13 as needed to identify all of the associated conditions that the patient has. 1) Type of diabetes The age of a patient is not the sole determining factor, though most type 1 diabetics develop the condition before reaching puberty. For this reason, type 1 diabetes mellitus is also referred to as juvenile diabetes. 2) Type of diabetes mellitus not documented If the type of diabetes mellitus is not documented in th

In [70]:
for res in results:
    print(len (res["content"]))

6396
1770
13863


In [74]:
coder_agent_res = '''### REFLECTION_START ###

### REASONING_START ###
1. Type 2 Diabetes:
- Patient has adult onset diabetes with nerve damage
- Requires neurological complication code
- Also has macular edema that was successfully treated
- Blood sugars are under control (<100)

2. Hypertension:
- Patient has essential hypertension
- Blood pressures are running normal

3. Asthma:
- Has mild persistent asthma since childhood
- No mention of complications or exacerbation

4. CKD:
- Mentioned as mild CKD
- No specific stage mentioned beyond "mild"

5. Eye Condition:
- Had macular edema in both eyes
- Successfully treated 2 years ago
- This aligns with diabetic macular edema resolved after treatment
### REASONING_END ###

### GRAPH_ANALYSIS_START ###
1. For E11 (Type 2 Diabetes):
- Followed path to E114 for neurological complications
- Followed path to E1140 as specific type not mentioned
- Followed path to E1137X3 for resolved macular edema (bilateral)

2. For I10:
- Direct billable code

3. For J45:
- Followed path to J453 (Mild persistent asthma)
- Then to J4530 (uncomplicated) as no exacerbation mentioned

4. For N18:
- Followed path to N182 as specifically mentioned as mild CKD

5. For H57:
- No specific eye disorder from this category is mentioned in the text
### GRAPH_ANALYSIS_END ###

### GUIDELINES_ANALYSIS_START ###
1. For Diabetes: Per guidelines section 4.a, "As many codes within a particular category as are necessary to describe all of the complications of the disease may be used."

2. For CKD: Per guidelines section 14.a.1, "Stage 2, code N18.2, equates to mild CKD"

3. For Hypertension: Per guidelines section 9.a.8, "Controlled hypertension" is coded with appropriate code from I10-I15.
### GUIDELINES_END ###

### REFLECTION_END ###'''

In [79]:
coder_agent_reasons = " "
# Regular expression to capture the list inside the square brackets
pattern = r"### REFLECTION_START ###\s*([^\]]+)\s*### REFLECTION_END ###"

# Search for the pattern
match = re.search(pattern, coder_agent_res)

if match:
    coder_agent_reasons = match.group(1).strip()

In [80]:
coder_agent_reasons

'### REASONING_START ###\n1. Type 2 Diabetes:\n- Patient has adult onset diabetes with nerve damage\n- Requires neurological complication code\n- Also has macular edema that was successfully treated\n- Blood sugars are under control (<100)\n\n2. Hypertension:\n- Patient has essential hypertension\n- Blood pressures are running normal\n\n3. Asthma:\n- Has mild persistent asthma since childhood\n- No mention of complications or exacerbation\n\n4. CKD:\n- Mentioned as mild CKD\n- No specific stage mentioned beyond "mild"\n\n5. Eye Condition:\n- Had macular edema in both eyes\n- Successfully treated 2 years ago\n- This aligns with diabetic macular edema resolved after treatment\n### REASONING_END ###\n\n### GRAPH_ANALYSIS_START ###\n1. For E11 (Type 2 Diabetes):\n- Followed path to E114 for neurological complications\n- Followed path to E1140 as specific type not mentioned\n- Followed path to E1137X3 for resolved macular edema (bilateral)\n\n2. For I10:\n- Direct billable code\n\n3. For J4